In [ ]:
import os
import re
from datetime import datetime
import pandas as pd
# ==========================================
# 設定
# ==========================================
SAMPLES_DIR = "/content/samples"  # 必要に応じてパスを変更してください
INVENTORY_PATH = os.path.join(SAMPLES_DIR, "inventory.xlsx")
ORDER_DIR = os.path.join(SAMPLES_DIR, "order_new")
PICKUP_PATH = os.path.join(SAMPLES_DIR, "pickup.xlsx")  # 必要に応じて変更
# 対象の7品目
TARGET_ITEMS = [
    "トマト",
    "キャベツ",
    "レタス",
    "白菜",
    "ほうれん草",
    "大根",
    "ニンジン",
]
def main():
  print("=== 野菜卸売業 受発注・在庫管理＆追加発注判定プログラム ===\n")
  # 1. 在庫管理表の読み込み
  if not os.path.exists(INVENTORY_PATH):
    print(f"エラー: 在庫管理表が見つかりません: {INVENTORY_PATH}")
    return
  inv_df = pd.read_excel(INVENTORY_PATH)
  inv_df["日付"] = pd.to_datetime(inv_df["日付"])
  # 2. order_newフォルダから注文ファイルをすべて読み込み、商品名単位で集約
  if os.path.exists(ORDER_DIR):
    order_files = [
        f
        for f in os.listdir(ORDER_DIR)
        if f.endswith(".xlsx") and not f.startswith("~$")
    ]
  else:
    order_files = []
  if not order_files:
    order_files = [
        f
        for f in os.listdir(SAMPLES_DIR)
        if f.startswith("order_") and f.endswith(".xlsx")
    ]
  if order_files:
    date_orders = {}  # {date_str: {商品名: 合計注文数}}
    for file_name in order_files:
      match = re.search(r"(\d{8})", file_name)
      if not match:
        continue
      date_str = match.group(1)
      file_path = os.path.join(ORDER_DIR, file_name)
      if not os.path.exists(file_path):
        file_path = os.path.join(SAMPLES_DIR, file_name)
      try:
        order_df = pd.read_excel(file_path)
      except Exception as e:
        print(f"エラー: ファイルの読み込みに失敗しました {file_name}: {e}")
        continue
      if date_str not in date_orders:
        date_orders[date_str] = {item: 0 for item in TARGET_ITEMS}
      # 注文表の列名（商品名）と行データを対比しながら集約
      for col in order_df.columns:
        item_name = str(col).strip()
        if item_name in TARGET_ITEMS:
          col_sum = order_df[col].sum(numeric_only=True)
          if pd.notna(col_sum):
            date_orders[date_str][item_name] += col_sum
    # 3. 在庫管理表の更新処理（商品名単位で減算）
    for date_str, item_orders in date_orders.items():
      target_date = pd.to_datetime(date_str, format="%Y%m%d")
      match_indices = inv_df[inv_df["日付"] == target_date].index
      prev_df = inv_df[inv_df["日付"] < target_date]
      base_row = (
          prev_df.sort_values("日付").iloc[-1]
          if not prev_df.empty
          else inv_df.iloc[0]
      )
      if not match_indices.empty:
        idx = match_indices[0]
        print(
            f"【更新】既存データの日付 ({target_date.strftime('%Y-%m-%d')})"
            " の在庫を商品名ごとに再計算します。"
        )
        for item in TARGET_ITEMS:
          if item in inv_df.columns:
            base_val = (
                base_row[item]
                if item in base_row and pd.notna(base_row[item])
                else 0
            )
            order_val = item_orders.get(item, 0)
            inv_df.loc[idx, item] = base_val - order_val
      else:
        print(
            f"【追加】新規日付 ({target_date.strftime('%Y-%m-%d')})"
            " の行を作成し、商品名ごとに最新日ベースから減算します。"
        )
        inv_df = inv_df.sort_values("日付").reset_index(drop=True)
        latest_row = inv_df.iloc[-1].copy()
        new_row = latest_row.copy()
        new_row["日付"] = target_date
        new_row["曜日"] = target_date.strftime("%a")
        for item in TARGET_ITEMS:
          if item in new_row.index:
            base_val = (
                latest_row[item] if pd.notna(latest_row[item]) else 0
            )
            order_val = item_orders.get(item, 0)
            new_row[item] = base_val - order_val
        new_df = pd.DataFrame([new_row])
        inv_df = pd.concat([inv_df, new_df], ignore_index=True)
    inv_df = inv_df.sort_values("日付").reset_index(drop=True)
    # 在庫管理表の保存
    # 日付は文字列に変換せず、datetime型のままExcelに保存する。
    # Excel上の表示形式を「2023-05-01 00:00:00」に設定する。
    with pd.ExcelWriter(
        INVENTORY_PATH,
        engine="openpyxl",
        datetime_format="yyyy-mm-dd hh:mm:ss"
    ) as writer:
      inv_df.to_excel(writer, index=False)

    print(f"\n在庫管理表を正常に更新・保存しました: {INVENTORY_PATH}\n")
  else:
    print("処理対象の注文ファイルがありません（在庫更新はスキップします）。\n")
  # 4. 在庫管理表を画面に表示
  print("=== 更新後の在庫管理表（直近データ） ===")
  inv_df_view = inv_df.copy()
  inv_df_view["日付"] = inv_df_view["日付"].dt.strftime("%Y-%m-%d %H:%M:%S")
  print(inv_df_view.tail(10).to_string(index=False))
  print("\n" + "=" * 50 + "\n")
  # ==========================================
  # 5. 7品目の追加発注判定処理（商品名対比）
  # ==========================================
  print("--- 【7品目の追加発注判定】 ---")
  if not os.path.exists(PICKUP_PATH):
    print(f"注意: ピックアップ（しきい値）ファイルが見つかりません: {PICKUP_PATH}")
    print("追加発注の判定をスキップします。\n")
    return
  pickup_df = pd.read_excel(PICKUP_PATH)
  # 商品名の表記ゆれ（全角/半角スペース、「人参」など）を統一する
  import unicodedata
  def normalize_item_name(value):
    if pd.isna(value):
      return ""
    name = unicodedata.normalize("NFKC", str(value))
    name = re.sub(r"\s+", "", name).replace("　", "")
    # よくある表記ゆれを統一
    aliases = {
        "人参": "ニンジン", "にんじん": "ニンジン", "ﾆﾝｼﾞﾝ": "ニンジン",
        "ほうれん草": "ほうれんそう", "ホウレンソウ": "ほうれんそう",
    }
    name = aliases.get(name, name)
    return name

  # pickup.xlsxの列名とデータ内容の両方から列を特定する
  columns = list(pickup_df.columns)
  col_item, col_threshold, col_add = None, None, None

  # まず、見出しに含まれるキーワードから候補を探す
  for col in columns:
    header = normalize_item_name(col)
    if col_item is None and any(k in header for k in ["商品", "品名", "品目", "野菜", "トマト", "キャベツ", "レタス", "白菜", "ほうれん草", "大根", "ニンジン"]):
      col_item = col
    if col_threshold is None and any(k in header for k in ["しきい値", "閾値", "下限", "基準在庫", "発注点", "最低在庫"]):
      col_threshold = col
    if col_add is None and any(k in header for k in ["追加量", "発注量", "発注数", "追加数量", "数量"]):
      col_add = col

  # 見出しが想定と異なる場合、実際のセル内容から商品名の列を探す
  target_names = {normalize_item_name(x) for x in TARGET_ITEMS}
  if col_item is None:
    best_col, best_score = None, 0
    for col in columns:
      values = pickup_df[col].dropna().astype(str)
      score = sum(normalize_item_name(v) in target_names for v in values)
      if score > best_score:
        best_col, best_score = col, score
    col_item = best_col

  # 数値列の候補。見出しで特定できない場合に使用する
  numeric_candidates = []
  for col in columns:
    if col == col_item:
      continue
    converted = pd.to_numeric(pickup_df[col], errors="coerce")
    if converted.notna().any():
      numeric_candidates.append(col)

  if col_threshold is None:
    # しきい値らしい見出しを優先し、なければ最初の数値列
    col_threshold = numeric_candidates[0] if numeric_candidates else None
  if col_add is None:
    # 追加発注量らしい見出しを優先し、なければ次の数値列
    remaining = [c for c in numeric_candidates if c != col_threshold]
    col_add = remaining[0] if remaining else None

  if col_item is None or col_threshold is None or col_add is None:
    print("エラー: pickup.xlsxから商品名・しきい値・追加発注量の列を特定できませんでした。")
    print("pickup.xlsxの列名とデータ内容を確認してください。")
    return

  # 最新の在庫量を確認（inventory.xlsxの最終行）
  inv_df = inv_df.sort_values("日付").reset_index(drop=True)
  latest_inventory = inv_df.iloc[-1]
  latest_date_str = latest_inventory["日付"].strftime("%Y-%m-%d %H:%M:%S")
  print(f"確認対象の最新在庫日付: {latest_date_str}\n")
  judgment_results = []

  # pickup_dfの各行を読み込み、商品名を正規化してTARGET_ITEMSと照合する
  for _, row in pickup_df.iterrows():
    raw_item_name = row.get(col_item)
    if pd.isna(raw_item_name):
      continue
    item_name = normalize_item_name(raw_item_name)
    # TARGET_ITEMS側も同じルールで照合する
    target_lookup = {normalize_item_name(x): x for x in TARGET_ITEMS}
    if item_name not in target_lookup:
      continue
    canonical_item = target_lookup[item_name]

    # inventory.xlsxの列名にも表記ゆれがある場合に備えて対応する列を探す
    inventory_col_lookup = {normalize_item_name(c): c for c in inv_df.columns}
    inventory_col = inventory_col_lookup.get(item_name)
    if inventory_col is None:
      print(f"注意: inventory.xlsxに {canonical_item} の在庫列が見つかりません。")
      continue

    # Excelから読み込んだ数値が文字列でも比較できるよう数値に変換する
    threshold = pd.to_numeric(row.get(col_threshold), errors="coerce")
    add_qty = pd.to_numeric(row.get(col_add), errors="coerce")
    current_stock = pd.to_numeric(latest_inventory.get(inventory_col), errors="coerce")

    if pd.isna(threshold) or pd.isna(add_qty) or pd.isna(current_stock):
      print(f"注意: {canonical_item} の在庫数・しきい値・追加発注量のいずれかが数値ではないため、判定をスキップします。")
      continue

    # しきい値を下回っているか判定（例: 18 < 40 の場合は True）
    is_below = current_stock < threshold
    judgment_results.append({
        "商品名": canonical_item,
        "最新在庫数": current_stock,
        "しきい値": threshold,
        "判定": "【要追加発注】" if is_below else "適正",
        "追加発注量": add_qty if is_below else 0,
    })
  # 結果を分かりやすくテーブル形式で画面表示
  if judgment_results:
    result_df = pd.DataFrame(judgment_results)
    print(result_df.to_string(index=False))
  else:
    print("対象となる商品としきい値データが一致しませんでした。")
  print("\n==========================================")

if __name__ == "__main__":
  main()


=== 野菜卸売業 受発注・在庫管理＆追加発注判定プログラム ===

【追加】新規日付 (2023-05-24) の行を作成し、商品名ごとに最新日ベースから減算します。

在庫管理表を正常に更新・保存しました: /content/samples/inventory.xlsx

=== 更新後の在庫管理表（直近データ） ===
                 日付  曜日  トマト  キャベツ  レタス  白菜  ほうれん草  大根  ニンジン
2023-05-06 00:00:00 Sat   43    92   96  40     66  50    37
2023-05-07 00:00:00 Sun   81    39   74  18    104  38    77
2023-05-08 00:00:00 Mon  104    83   29  33     72  56    55
2023-05-09 00:00:00 Tue   31    36   84  35     92  44    78
2023-05-10 00:00:00 Wed   92    75   32  29     43  22    62
2023-05-11 00:00:00 Thu   61    29   54  52     33  78    92
2023-05-12 00:00:00 Fri   55    40   60  15     27  43    29
2023-05-13 00:00:00 Sat   49   103   41  10     35  33    58
2023-05-14 00:00:00 Sun   91    73  103  84     75  48    50
2023-05-24 00:00:00 Wed   60    52   61  59     52  33    18


--- 【7品目の追加発注判定】 ---
確認対象の最新在庫日付: 2023-05-24 00:00:00

対象となる商品としきい値データが一致しませんでした。

